# Batched power-system simulation with PyTorch

Build a transmission-network simulator using tensor operations, batched linear solves, sequential state updates, and forward-mode automatic differentiation. No neural network is needed.

**Prerequisites:** basic Python, matrix multiplication, and derivatives. Work through the eight exercises in order; later sections depend on earlier ones. Each exercise is followed by checks. Reference answers are in \`../solutions/reference.py\`; see the README for instructions.

## What this project models
A synthetic five-bus network illustrates ideas relevant to transmission-system operators. DC power flow approximates an **AC network**: voltage magnitudes near 1 per unit, small angle differences, negligible resistance, and no reactive-power or losses. It does not establish voltage security or dynamic stability.

Our sequential simulation consists of steady-state network solves coupled through storage energy. It is not an electromechanical transient simulation. The aim is to understand what can be batched and how sensitivities propagate, before adding an optimiser.


In [ ]:
import math
import time
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)
DEVICE = torch.device("cpu")  # Start on CPU; try CUDA later.
DTYPE = torch.float64
BASE_MVA = 100.0
torch.manual_seed(7)

def tensor(values):
    return torch.tensor(values, dtype=DTYPE, device=DEVICE)

print("PyTorch:", torch.__version__, "| device:", DEVICE)


## 1. Network, units, and signs

Positive bus injection means generation exceeds demand. Each line's orientation defines its flow sign, not a restriction on flow direction.

Reactances are per unit on a common 100 MVA base; 100 MW corresponds to 1 pu of active power. Angles are radians. Bus 0 is both our reference-angle bus and the bus assigned the balancing injection. These are conceptually separate choices.

| Line | From | To | Reactance [pu] | Limit [MW] |
|---|---:|---:|---:|---:|
| 0 | 0 | 1 | 0.10 | 90 |
| 1 | 0 | 2 | 0.20 | 80 |
| 2 | 1 | 2 | 0.15 | 60 |
| 3 | 1 | 3 | 0.10 | 70 |
| 4 | 2 | 3 | 0.12 | 70 |
| 5 | 2 | 4 | 0.10 | 80 |
| 6 | 3 | 4 | 0.15 | 60 |


In [ ]:
N = 5
ends = torch.tensor(
    [[0, 1], [0, 2], [1, 2], [1, 3], [2, 3], [2, 4], [3, 4]],
    dtype=torch.long, device=DEVICE,
)
x = tensor([0.10, 0.20, 0.15, 0.10, 0.12, 0.10, 0.15])
limits = tensor([90, 80, 60, 70, 70, 80, 60]) / BASE_MVA
L = len(x)
b = 1 / x

positions = [(0, 1), (1, 2), (1, 0), (2, 2), (2, 0)]
fig, ax = plt.subplots(figsize=(7, 4))
for k, (i, j) in enumerate(ends.cpu().tolist()):
    xi, yi = positions[i]
    xj, yj = positions[j]
    ax.plot([xi, xj], [yi, yj], color="gray")
    ax.text((xi+xj)/2, (yi+yj)/2, f"line {k}", fontsize=9)
for i, (px, py) in enumerate(positions):
    ax.scatter(px, py, s=300, color="lightblue", zorder=3)
    ax.text(px, py, str(i), ha="center", va="center", zorder=4)
ax.set_aspect("equal")
ax.axis("off")
plt.show()


## 2. Derive the network equations

The incidence matrix \(A\in\mathbb R^{L\times N}\) has +1 at a line's from-bus and −1 at its to-bus. Thus \(A\theta\) contains angle differences.

With \(b_\ell=1/x_\ell\),

\[
f=\operatorname{diag}(b)A\theta,\qquad
p=A^\mathsf T f=B\theta,\qquad
B=A^\mathsf T\operatorname{diag}(b)A.
\]

\(B\) is singular because adding a constant to all angles changes no flows. Fix \(\theta_0=0\), remove row and column 0, and solve

\[
B_{\rm red}\theta_{\rm red}=p_{\rm red}.
\]

For a connected network with positive reactances, \(B_{\rm red}\) is symmetric positive definite.

### Exercise 1 — Build the network matrices
Implement the incidence and nodal matrices. Scale rows of \(A\) with broadcasting instead of constructing a diagonal matrix.

**Explain:** why does each row of \(B\) sum to zero? What fails if the network splits into islands?


In [ ]:
def incidence(ends, n):
    """Return [L, n]; +1 at from-bus, -1 at to-bus."""
    raise NotImplementedError("Exercise 1")

def nodal_matrix(A, b):
    """Return [N, N]."""
    raise NotImplementedError("Exercise 1")


In [ ]:
A = incidence(ends, N)
B = nodal_matrix(A, b)
Bred = B[1:, 1:]
torch.testing.assert_close(A.sum(-1), torch.zeros(L, device=DEVICE))
torch.testing.assert_close(B, B.T)
torch.testing.assert_close(B.sum(-1), torch.zeros(N, device=DEVICE))
assert torch.linalg.eigvalsh(Bred).min() > 0
assert torch.linalg.matrix_rank(B).item() == N - 1
print("Network checks passed.")


## 3. Batch independent scenarios

Keep buses or lines on the last dimension:

| Quantity | Shape |
|---|---|
| One injection vector | \`[N]\` |
| Scenario batch | \`[S, N]\` |
| Independent scenario/time snapshots | \`[S, T, N]\` |
| Corresponding flows | \`[S, T, L]\` |

For row-vector storage, \(f=(\theta A^\mathsf T)\odot b\) and \(p=fA\).

### Exercise 2 — Batched DC power flow
Prepend the balancing injection \(p_0=-\sum_{i=1}^{N-1}p_i\).
Implement a solver accepting arbitrary leading batch dimensions.
Flatten these dimensions, arrange scenarios as columns of a multiple-right-hand-side system, solve, and reshape.

Use \`torch.linalg.solve\`, not an explicit inverse. Reject unbalanced inputs rather than silently modifying them in the solver.


In [ ]:
def balance_slack(p_non_slack):
    """Input [..., N-1]; return balanced [..., N]."""
    raise NotImplementedError("Exercise 2")

def dc_power_flow(p, A, b):
    """Return theta [..., N] and flow [..., L], reference bus 0."""
    raise NotImplementedError("Exercise 2")


In [ ]:
p = balance_slack(tensor([20, -60, -30, 10]) / BASE_MVA)
theta, flow = dc_power_flow(p, A, b)
print("Angles [rad]:", theta)
print("Line flows [MW]:", flow * BASE_MVA)
torch.testing.assert_close(flow @ A, p)
torch.testing.assert_close(theta[0], tensor(0))

g = torch.Generator(device=DEVICE).manual_seed(17)
p_batch = balance_slack(
    0.25 * torch.randn(32, N-1, generator=g, device=DEVICE)
)
theta_batch, flow_batch = dc_power_flow(p_batch, A, b)
torch.testing.assert_close(flow_batch @ A, p_batch)
flow_loop = torch.stack([dc_power_flow(q, A, b)[1] for q in p_batch])
torch.testing.assert_close(flow_batch, flow_loop)
_, f_grid = dc_power_flow(p_batch.reshape(4, 8, N), A, b)
assert f_grid.shape == (4, 8, L)
torch.testing.assert_close(f_grid.reshape(32, L), flow_batch)

# Analytic fixture: 50 MW transfer across x=0.1.
theta2, f2 = dc_power_flow(tensor([0.5, -0.5]), tensor([[1, -1]]), tensor([10]))
torch.testing.assert_close(theta2, tensor([0, -0.05]))
torch.testing.assert_close(f2, tensor([0.5]))
try:
    dc_power_flow(tensor([0.5, 0]), tensor([[1, -1]]), tensor([10]))
except ValueError:
    pass
else:
    raise AssertionError("Unbalanced injections must be rejected.")
print("Power-flow checks passed.")


## 4. Reuse the network factorisation

For fixed topology and reactances only the right-hand sides change.
Factorise \(B_{\rm red}=CC^\mathsf T\) once and reuse it. This benefits CPUs too.

### Exercise 3 — Prepare a reusable solver
Use \`torch.linalg.cholesky\` once in \`prepare_solver\`, then \`torch.cholesky_solve\` in the returned callable. Cache \(H=\operatorname{diag}(b)A\) too.

The inner callable assumes balanced inputs, allowing us to avoid a GPU-to-CPU synchronisation for a Python boolean balance check at every step. Validate at the scenario boundary.

Rebuild the factorisation when topology or reactances change. This exercise differentiates injections and controls, not cached network parameters.


In [ ]:
def prepare_solver(A, b):
    """Return solve(p) -> (theta, flow), with a cached factorisation."""
    raise NotImplementedError("Exercise 3")


In [ ]:
solve = prepare_solver(A, b)
theta_fast, flow_fast = solve(p_batch)
torch.testing.assert_close(theta_fast, theta_batch)
torch.testing.assert_close(flow_fast, flow_batch)


## 5. Time-dependent state, parallel scenarios

Without state coupling, all time snapshots can be solved together.
Storage creates a dependency across time: advance sequentially while
operating on all scenarios in parallel.

Place storage at bus 4. Positive \(u_t\) discharges into the network.
For ideal storage,

\[
e_{t+1}=e_t-u_t\Delta t.
\]

Energy is measured in base-MW-hours: 1 unit equals 100 MWh. The policy is

\[
u_t=\operatorname{clip}\big(k(w_{\rm target}-w_t),
u_{\min}(e_t),u_{\max}(e_t)\big),
\]

\[
u_{\min}=\max(-P_{\max},(e_t-E_{\max})/\Delta t),\qquad
u_{\max}=\min(P_{\max},e_t/\Delta t).
\]

It respects energy and power bounds, but does not optimise congestion.

### Exercise 4 — Sequential storage simulation
Loop over time, with each step batched over scenarios. Initialise energy at half capacity. Add wind and storage injection at bus 4 and recompute the balancing injection.

Return flows \`[S,T,L]\`, energy \`[S,T+1]\`, and actions \`[S,T]\`.
Use lists and \`torch.stack\` to preserve differentiation. Use the supplied wind argument so later sensitivity experiments can perturb it.


In [ ]:
S, T = 128, 24
dt = 1.0
capacity = 1.0
power_limit = 0.25
wind_target = 0.30
hours = torch.arange(T, dtype=DTYPE, device=DEVICE)
daily = 1 + 0.15 * torch.sin(2 * math.pi * hours / T)
g = torch.Generator(device=DEVICE).manual_seed(23)
noise = torch.randn(S, T, 4, generator=g, device=DEVICE)
demand = (
    tensor([0.25, 0.50, 0.35, 0.20])[None, None, :]
    * daily[None, :, None] * (1 + 0.05 * noise)
).clamp_min(0)
wind = (
    0.30 + 0.15 * torch.sin(2 * math.pi * hours / T)[None, :]
    + 0.06 * torch.randn(S, T, generator=g, device=DEVICE)
).clamp(0, 0.65)
generation = tensor([0.30, 0, 0, 0])
wind_bus = tensor([0, 0, 0, 1])
p_non_slack_series = generation - demand + wind[..., None] * wind_bus
p_snapshots = balance_slack(p_non_slack_series)
_, independent_flows = solve(p_snapshots)


In [ ]:
def rollout(gain, wind_input):
    """Return batched flows, energy history, and storage actions."""
    raise NotImplementedError("Exercise 4")


In [ ]:
gain = tensor(0.4)
flows, energies, actions = rollout(gain, wind)
assert flows.shape == (S, T, L)
assert energies.shape == (S, T+1)
assert actions.shape == (S, T)
assert energies.min() >= -1e-10
assert energies.max() <= capacity + 1e-10
assert actions.abs().max() <= power_limit + 1e-10
torch.testing.assert_close(energies[:, 1:] - energies[:, :-1], -actions * dt)
expected_non_slack = p_non_slack_series + actions[..., None] * wind_bus
torch.testing.assert_close(flows @ A, balance_slack(expected_non_slack))
f0, e0, u0 = rollout(tensor(0), wind)
torch.testing.assert_close(f0, independent_flows)
torch.testing.assert_close(u0, torch.zeros_like(u0))

fig, axes = plt.subplots(2, 1, figsize=(9, 6))
axes[0].plot(energies[:5].T.detach().cpu() * BASE_MVA)
axes[0].set_ylabel("Stored energy [MWh]")
axes[1].plot((flows.abs() / limits).amax(-1)[:5].T.detach().cpu())
axes[1].axhline(1, color="red", linestyle="--")
axes[1].set_ylabel("Maximum line loading / limit")
axes[1].set_xlabel("Hour")
plt.tight_layout()
plt.show()


## 6. Dual tensors and directional sensitivities

For \(y=F(z)\), forward-mode AD computes \( \dot y=J_F(z)\dot z \).
A dual tensor carries a value and a tangent; \`torch.func.jvp\` handles
this conveniently.

### Exercise 5 — Sensitivity to an injection transfer
Increase injection at bus 4 and decrease injection at bus 0 by the same amount. Compute the flow JVP. Check it against a direct solve of the tangent injections and a central finite difference.

For fixed-network DC power flow,

\[
B_{\rm red}\dot\theta_{\rm red}=\dot p_{\rm red}.
\]

A tangent of 1 pu is a 100 MW direction. The numerical sensitivity in
pu-flow/pu-injection equals MW-flow/MW-injection on this common base.

**Explain:** why is the tangent identical across base scenarios?
Would the same be true in AC power flow?


In [ ]:
from torch.func import jvp

direction = torch.zeros_like(p_batch)
direction[:, 4] = 1
direction[:, 0] = -1

def flows_only(p):
    return solve(p)[1]


In [ ]:
# Use jvp(flows_only, (p_batch,), (direction,)).
primal_flows, flow_tangent = None, None
raise NotImplementedError("Exercise 5")


In [ ]:
torch.testing.assert_close(primal_flows, flow_batch)
torch.testing.assert_close(flow_tangent, solve(direction)[1])
eps = 1e-5
fd = (flows_only(p_batch + eps*direction) - flows_only(p_batch - eps*direction)) / (2*eps)
torch.testing.assert_close(flow_tangent, fd, rtol=1e-7, atol=1e-9)
torch.testing.assert_close(flow_tangent, flow_tangent[:1].expand_as(flow_tangent))


### Exercise 6 — Sensitivities through sequential state

Compute every line flow's sensitivity to the scalar policy gain \(k\).
For \(e_{t+1}=F(e_t,k,w_t)\), its tangent propagates through time:

\[
\dot e_{t+1}=F_e\dot e_t+F_k\dot k+F_w\dot w_t.
\]

Forward mode suits one input direction with many outputs. Reverse mode
usually suits many control inputs with one scalar objective.

Also differentiate the mean squared line-loading score below. This
score is not a security constraint or an economic cost.

Clipping is piecewise differentiable. At switching boundaries a classical
derivative may not exist. Check finite differences with several step
sizes: they may cross different operating regimes.

**Explain:** why can sensitivity be zero under saturation? Why can one
action affect later hours? What changes if all hourly actions are inputs?


In [ ]:
def gain_to_flows(k):
    return rollout(k, wind)[0]

def loading_score(k):
    return ((gain_to_flows(k) / limits) ** 2).mean()


In [ ]:
# Use a scalar tangent torch.ones_like(gain) for both JVPs.
base_flows, gain_tangent = None, None
score, score_tangent = None, None
raise NotImplementedError("Exercise 6")


In [ ]:
expected = (2 * base_flows * gain_tangent / limits.square()).mean()
torch.testing.assert_close(score_tangent, expected)
errors = []
for eps in [1e-3, 1e-4, 1e-5]:
    fd = (gain_to_flows(gain+eps) - gain_to_flows(gain-eps)) / (2*eps)
    error = (
        torch.linalg.vector_norm(fd-gain_tangent)
        / torch.linalg.vector_norm(gain_tangent).clamp_min(1e-12)
    ).item()
    errors.append(error)
    print(f"epsilon={eps:g}; relative JVP error={error:.3e}")
# This fixture should have a sufficiently local finite difference.
assert errors[-1] < 1e-3

plt.figure(figsize=(9, 4))
plt.plot(gain_tangent[0].detach().cpu())
plt.xlabel("Hour")
plt.ylabel("Line-flow sensitivity [pu / gain]")
plt.title("Scenario 0: sensitivity to policy gain")
plt.show()


## 7. Scenario-dependent topology

An outage changes \(B\). Reusing the pre-outage factorisation gives
incorrect results.

### Exercise 7 — Single-line outage screening
For every outage, set its susceptance to zero and prepare a new solver.
Check graph connectivity first: traverse remaining lines starting at
bus 0. Return \`NaN\` for islanding cases.

For connected cases, solve the same balanced injections, verify the
outaged line has zero flow, and calculate maximum absolute loading.
Return shape \`[L,S]\`.

This is fixed-injection DC screening. It ignores corrective actions,
emergency ratings, reactive power, and stability. It does not establish
full N−1 security. Keep islanding results visible rather than discarding
them from a worst-case summary.

**Extension:** group many scenario/outage combinations by topology to
reuse factorisations.


In [ ]:
def connected_after_outage(ends, n, outage):
    raise NotImplementedError("Exercise 7")

def outage_loading(p_batch, A, b, limits):
    raise NotImplementedError("Exercise 7")


In [ ]:
outage_result = outage_loading(p_batch, A, b, limits)
assert outage_result.shape == (L, len(p_batch))
assert torch.isfinite(outage_result).all()  # This mesh survives each outage.
chain = torch.tensor([[0, 1], [1, 2]], device=DEVICE)
assert not connected_after_outage(chain, 3, 0)
assert not connected_after_outage(chain, 3, 1)

plt.figure(figsize=(9, 4))
plt.imshow(outage_result.detach().cpu(), aspect="auto")
plt.xlabel("Scenario")
plt.ylabel("Outaged line")
plt.colorbar(label="Maximum line loading / limit")
plt.show()


## 8. Benchmark batching fairly

### Exercise 8 — Scenario loop versus one batched call
Use the same cached solver in both methods. Benchmark batch sizes
1, 32, 256, and 2048, checking equivalent outputs.

Warm up, measure medians, and synchronise CUDA before and after timing.
Keep tensors resident on the device for this compute-only benchmark.
If transfers and factorisation occur in your application, measure them
separately and include them in an end-to-end comparison.

This tiny dense network demonstrates batching; it cannot establish a
GPU advantage for large, sparse transmission models. Dense matrix
storage scales quadratically, and sparse solver support matters.

Change \`DEVICE\` in the setup cell and restart/run the notebook to
benchmark CUDA. Rebuild all network tensors and cached factors.

**Write up:** hardware, PyTorch version, dtype, timings, speedups, and
what costs your measurement includes. Explain when batching helps and
when memory or synchronisation becomes a bottleneck.


In [ ]:
def synchronise():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)

def median_time(fn, warmup=3, repeats=7):
    for _ in range(warmup):
        fn()
    synchronise()
    timings = []
    for _ in range(repeats):
        synchronise()
        start = time.perf_counter()
        fn()
        synchronise()
        timings.append(time.perf_counter() - start)
    return sorted(timings)[len(timings)//2]


In [ ]:
def benchmark():
    # Generate balanced inputs for each size.
    # Compare torch.stack([solve(row)[1] for row in q]) with solve(q)[1].
    # Use median_time; report milliseconds and speedup.
    raise NotImplementedError("Exercise 8")


In [ ]:
benchmark()


## Next steps and European context

Extend this project with storage efficiency, ramp-limited generation,
wind perturbations at individual hours, or topology-grouped scenarios.
Compare forward JVPs with reverse-mode gradients of a scalar score.

For AC power flow \(g(x,p)=0\), implicit differentiation gives

\[
g_x\dot x=-g_p\dot p.
\]

Reusing the final Newton Jacobian factorisation can avoid differentiating
every solver iteration. Convergence, conditioning, and changes in
generator-limit regimes require care.

Physical injection sensitivities connect to PTDFs. European zonal
trade sensitivities additionally require assumptions about how changes
in a bidding zone's net position translate into bus injections
(generation shift keys). A bus-level simulator alone does not model
flow-based market coupling.

These automatic-differentiation duals are distinct from the Lagrange
multipliers of constrained optimisation.

## Reading
- [PyTorch forward-mode AD](https://docs.pytorch.org/tutorials/intermediate/forward_ad_usage.html)
- [torch.func.jvp](https://docs.pytorch.org/docs/stable/generated/torch.func.jvp.html)
- [MATPOWER](https://matpower.org/documentation/)
- [pandapower](https://pandapower.readthedocs.io/)
- [PyPSA](https://docs.pypsa.org/)
- [ENTSO-E](https://www.entsoe.eu/)
